### Preparation

In [ ]:
import pandas as pd
import numpy as np
import os, sys
import seaborn as sns
import matplotlib.pyplot as plt
import librosa

%matplotlib inline

### Data

In [ ]:
# Get list of participants
participants = ['292711', '783520', '957921', 'djtosh', 'fosmvm', 'fubqui', 
                'fxdrni', 'glbbwi', 'kdkycn', 'lfraeu', 'nefwgv', 'nqqivt',
                'onbana', 'sdwmkv', 'uypoyr', 'vroghc', 'wxbeka', 'yenswd',
                'ytzemq']

In [ ]:
# Specify the paths to the VR csv files.
files = []
for p in participants:
    # Open a file
    path = f"C:/Users/monik/Documents/Paris/Internship/Data-Collection/Extracted-Data/VR/{p}-VR"
    dirs = os.listdir( path )

    # This would print all the files and directories
    for file in dirs:
        files.append(file)

In [ ]:
# Read in the csv files
full = []
for p in participants:
    path = f"C:/Users/monik/Documents/Paris/Internship/Data-Collection/Extracted-Data/VR/{p}-VR"
    for file in files:
        name = file.split('_')
        if name[0] == p:
            full.append(f'{path}/{file}')

In [ ]:
df = pd.DataFrame()
for i in full:
    df1 = pd.read_csv(i, sep=',', names=['TIME_FRAME', 'f0', 'DURATION'])
    name = i.split('_')
    n = name[0].split('/')
    n0 = name[-1].split('.')
    df1['PARTICIPANT_ID'] = n[-1]
#     df1['TRIAL_TYPE'] = name[1]
#     df1['TRIAL_NO'] = n0[0]
    df1['TRIAL'] = f'{name[1]}_{n0[0]}'
    df = pd.concat([df, df1], axis=0)

In [ ]:
# Check what you made
# df[df['PARTICIPANT_ID'] == '292711']
df.head()

In [ ]:
# Check more
print(df['PARTICIPANT_ID'].unique())
print(df['TRIAL'].unique())
print(len(df['PARTICIPANT_ID'].unique()))
print(df[df['PARTICIPANT_ID']=='fosmvm'])

### Mean & Median

In [ ]:
# Compute mean pitch from all warm-up trials per participant
means = []
for p in participants:
    mean = df[df['PARTICIPANT_ID'] == p]['f0'].mean()
    means.append(round(mean, 2))

# Compute median pitch from all warm-up trials per participant
medians = []
for p in participants:
    median = df[df['PARTICIPANT_ID'] == p]['f0'].median()
    medians.append(round(median, 2))

### Min & Max

In [ ]:
# Compute mean of max & min per trial
mean_max = []
mean_min = []

for p in participants:
    data = df[df['PARTICIPANT_ID'] == p]
    
    max_pitch = []
    min_pitch = []
    
    pma1 = []
    pma2 = []
    pma3 = []
    
    pmi1 = []
    pmi2 = []
    pmi3 = []
    
    for index, row in data.iterrows():
        if row['TRIAL'] == 'MAX_1':
            pma_1 = row['f0']
            pma1.append(pma_1)
        elif row['TRIAL'] == 'MAX_2':
            pma_2 = row['f0']
            pma2.append(pma_2)
        elif row['TRIAL'] == 'MAX_3':
            pma_3 = row['f0']
            pma3.append(pma_3)
            
        if row['TRIAL'] == 'MIN_1':
            pmi_1 = row['f0']
            pmi1.append(pmi_1)
        elif row['TRIAL'] == 'MIN_2':
            pmi_2 = row['f0']
            pmi2.append(pmi_2)
        elif row['TRIAL'] == 'MIN_3':
            pmi_3 = row['f0']
            pmi3.append(pmi_3)
            
    # Append the max pitch for each the warm-up trial per participant
    max_pitch.append(max(pma1))
    max_pitch.append(max(pma2))
    max_pitch.append(max(pma3))
    
    # Append the min pitch
    min_pitch.append(min(pmi1))
    min_pitch.append(min(pmi2))
    min_pitch.append(min(pmi3))

    MAXM = round((sum(max_pitch) / len(max_pitch)),2)
    MINM = round((sum(min_pitch) / len(min_pitch)),2)
    mean_max.append(round(MAXM, 2))
    mean_min.append(round(MINM, 2))

In [ ]:
# Construct the dataframe
VRF = pd.DataFrame()
VRF['PARTICIPANT_ID'] = participants
VRF['MEAN_f0'] = means
VRF['MEDIAN_f0'] = medians
VRF['MAX_f0'] = mean_max
VRF['MIN_f0'] = mean_min
VRF['RANGE_f0'] = np.array(mean_max) - np.array(mean_min)

# Look at it
VRF

In [ ]:
# Transform f0 columns to MIDI
VRM = pd.DataFrame()
VRM['MEAN_MIDI'] = librosa.core.hz_to_midi(means)
VRM['MEDIAN_MIDI'] = librosa.core.hz_to_midi(medians)
VRM['MAX_MIDI'] = librosa.core.hz_to_midi(mean_max)
VRM['MIN_MIDI'] = librosa.core.hz_to_midi(mean_min)
VRM['RANGE_MIDI'] = VRM['MAX_MIDI'] - VRM['MIN_MIDI']

# Round all MIDI values in MIDI columns
VR_MIDI = VRM.iloc[:, [*range(-5,0)]].round(decimals = 2)

# Look
VR_MIDI

In [ ]:
# Concatenate
VR = pd.concat([VRF, VR_MIDI], axis=1)

# See
VR

### Output

In [ ]:
# Save to csv
VR.to_csv("VR_modified_all.csv")